# Výsledky simulácie pre článok

Zozbiera výsledky bežiaceho modelu, porovná ich s číslami z DP a **uloží celú sadu**
do `simulation_for_article/<VARIANT>/`.

Ten istý notebook sa púšťa dvakrát — stačí prepnúť `VARIANT`:

| beh | VARIANT | model |
|---|---|---|
| 1. | `base_model` | pôvodný, bez zmien |
| 2. | `updated_version_1` | po úpravách z kamdojizdime |

## Predpoklady

```bash
pip install requests pandas pyarrow folium
docker run -d --name sim-base-model -p 8000:8000 simulation-brno
```

Plán a kontext: [`../simulation_for_article/PLAN.md`](../simulation_for_article/PLAN.md)

In [1]:
# ═══ NASTAVENIE ═══════════════════════════════════════════════════════
VARIANT   = "base_model"          # alebo "updated_version_1"
API_URL   = "http://localhost:8000"
POPIS     = "pôvodný model, bez zmien"   # ide do run_meta.json

SCENAR_ULICA     = "Jihlavská"    # hlavný scenár
SCENAR_NULL_LINK = 62             # hrana s ~0 objemom → odhad šumu
# ══════════════════════════════════════════════════════════════════════

import json, subprocess, time, hashlib
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False

OUT = Path("..") / "simulation_for_article" / VARIANT
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

M = {}          # sem sa zbierajú všetky metriky
print(f"variant : {VARIANT}")
print(f"ukladám : {OUT.resolve()}")


variant : base_model
ukladám : /Users/magdalena.ondruskova/git/personal/Traffic-sim-backend/simulation_for_article/base_model


In [2]:
def get_json(cesta, **params):
    """GET, ktorý nespadne — vráti None a vypíše dôvod."""
    try:
        r = requests.get(f"{API_URL}{cesta}", params=params, timeout=180)
    except requests.exceptions.ConnectionError:
        print(f"  ✗ {cesta}: API nebeží")
        return None
    if r.status_code != 200:
        det = ""
        try:
            det = str(r.json().get("detail", ""))[:120]
        except Exception:
            det = r.text[:120]
        print(f"  ✗ {cesta}: {r.status_code} {det}")
        return None
    return r.json()


def geojson_to_df(gj):
    if not gj:
        return pd.DataFrame()
    rows = []
    for f in gj.get("features", []):
        p = dict(f.get("properties") or {})
        g = f.get("geometry") or {}
        t = g.get("type")
        if t == "LineString":
            p["_coords"] = g["coordinates"]
        elif t == "MultiLineString":
            p["_coords"] = [c for part in g["coordinates"] for c in part]
        else:
            p["_coords"] = None
        rows.append(p)
    return pd.DataFrame(rows)


def uloz_json(obj, meno):
    p = OUT / meno
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  → {p.name}")


meta = get_json("/api/meta")
if meta is None:
    raise SystemExit("API nebeží — spusti kontajner a pusti bunku znova.")
print("✅", meta.get("place_name"), "| center", meta.get("map_center"))

✅ Brno, Czechia | center {'lat': 49.201834914660175, 'lng': 16.58811554801459}


---
## 1. Reporty z pipeline

Surové JSON sa uložia bez úprav, aby bola k dispozícii úplná stopa.

In [3]:
validacia  = get_json("/api/reports/validation")
kalibracia = get_json("/api/reports/calibration")
od_sumar   = get_json("/api/reports/od-summary")
siet_sumar = get_json("/api/reports/network")      # môže chýbať, netreba

for obj, meno in [(validacia, "validation_report.json"),
                  (kalibracia, "calibration_report.json"),
                  (od_sumar, "od_summary.json"),
                  (siet_sumar, "network_summary.json")]:
    if obj is not None:
        uloz_json(obj, meno)

  → validation_report.json
  → calibration_report.json
  → od_summary.json
  → network_summary.json


---
## 2. Sieť a zóny — porovnanie s DP

In [4]:
# všetky linky (bez filtra) — potrebné pre VHT/VKT aj pre počty
linky = geojson_to_df(get_json("/api/links", min_volume=0))
uzly  = geojson_to_df(get_json("/api/nodes"))
zony  = geojson_to_df(get_json("/api/zones"))

M["siet"] = {
    "hrany": int(len(linky)),
    "uzly": int(len(uzly)),
    "zony_taz": int(len(zony)),
}
if "is_external" in zony.columns:
    ext = pd.to_numeric(zony["is_external"], errors="coerce").fillna(0).astype(int)
    M["siet"]["zony_externe"] = int(ext.sum())
    M["siet"]["zony_interne"] = int((ext == 0).sum())

DP_SIET = {"hrany": 36105, "uzly": 32539, "zony_taz": 92,
           "zony_interne": 87, "zony_externe": 5}

porov_siet = pd.DataFrame([
    {"veličina": k, "model": M["siet"].get(k), "DP": v,
     "rozdiel": (M["siet"].get(k) - v) if M["siet"].get(k) is not None else None}
    for k, v in DP_SIET.items()
])
print("SIEŤ A ZÓNY")
display(porov_siet)

  ✗ /api/nodes: 500 Internal Server Error
SIEŤ A ZÓNY


,veličina,model,DP,rozdiel
0,hrany,36435,36105,330
1,uzly,0,32539,-32539
2,zony_taz,69,92,-23
3,zony_interne,64,87,-23
4,zony_externe,5,5,0


---
## 3. Dopyt — OD matica po segmentoch

In [5]:
if od_sumar:
    seg = od_sumar.get("segments", {}) or {}
    M["dopyt"] = {
        "od_spolu":            seg.get("combined_daily"),
        "commuting":           seg.get("commuting"),
        "other":               seg.get("other"),
        "external_local":      seg.get("external_local"),
        "external_through":    seg.get("external_through_total"),
        "external_spolu":      seg.get("external_total"),
    }

    DP_DOPYT = {"od_spolu": 641622, "other": 94147,
                "external_local": 130000, "external_through": 36610}

    porov_dopyt = pd.DataFrame([
        {"segment": k, "model": M["dopyt"].get(k), "DP": DP_DOPYT.get(k),
         "rozdiel_%": (round((M["dopyt"][k] - DP_DOPYT[k]) / DP_DOPYT[k] * 100, 1)
                       if k in DP_DOPYT and M["dopyt"].get(k) else None)}
        for k in M["dopyt"]
    ])
    print("DOPYT [voz./deň]")
    display(porov_dopyt)
else:
    print("od_summary.json nedostupný")

DOPYT [voz./deň]


,segment,model,DP,rozdiel_%
0,od_spolu,556040.0,641622.0,-13.3
1,commuting,297113.0,NaN,NaN
2,other,93035.0,94147.0,-1.2
3,external_local,130000.0,130000.0,0.0
4,external_through,35891.0,36610.0,-2.0
5,external_spolu,165891.0,NaN,NaN


---
## 4. Kalibrácia a validácia

**Pozor na dve sady čísel.** `calibration_reference` sú dáta, ktoré ODME videla
(65 % CSD úsekov), `holdout_validation` sú tie, ktoré nevidela (35 %). Verdikt sa
berie z holdoutu. DP zrejme uvádza kalibračnú časť — porovnávaj správny stĺpec.

In [6]:
if validacia:
    b  = validacia.get("benchmarks", {}) or {}
    cf = b.get("calibration_fit", {}) or {}
    hv = b.get("holdout_validation", {}) or {}
    cr = validacia.get("calibration_reference", {}) or {}

    M["kalibracia"] = {
        "n":          cr.get("n"),
        "matched":    cr.get("matched"),
        "r2":         cf.get("r2"),
        "slope":      cf.get("slope"),
        "pct_rmse":   cf.get("pct_rmse"),
        "bias_pct":   cf.get("bias_abs_pct"),
    }
    M["holdout"] = {
        "r2":         hv.get("r2"),
        "slope":      hv.get("slope"),
        "pct_rmse":   hv.get("pct_rmse"),
        "bias_pct":   hv.get("bias_abs_pct"),
        "adekvatnost": b.get("holdout_adequacy"),
        "prijate":     hv.get("overall_pass"),
    }
    M["screenlines"] = {
        "max_odchylka_pct": b.get("screenline_max_error_pct"),
        "porovnanych":      b.get("screenline_n_compared"),
        "vylucenych":       b.get("screenline_n_excluded"),
    }
    M["verdikt"] = {
        "zdroj":       b.get("verdict_source"),
        "overall_pass": b.get("overall_pass"),
        "quality_gates": validacia.get("quality_gates"),
    }

    DP_FIT = {"r2": 0.83, "slope": 0.94, "pct_rmse": 40.4, "bias_pct": 8.6}
    CIEL   = {"r2": "≥ 0.80", "slope": "0.85–1.15",
              "pct_rmse": "≤ 35", "bias_pct": "≤ 15"}

    tab = pd.DataFrame([
        {"metrika": k,
         "kalibračná": M["kalibracia"].get(k),
         "holdout": M["holdout"].get(k),
         "DP §4": DP_FIT.get(k),
         "cieľ": CIEL.get(k)}
        for k in ("r2", "slope", "pct_rmse", "bias_pct")
    ])
    print("ZHODA S POZOROVANÍM")
    display(tab)
    print(f"n kalibračných bodov : {M['kalibracia']['n']}  "
          f"(adekvátnosť holdoutu: {M['holdout']['adekvatnost']})")
    print(f"screenlines          : max odchýlka {M['screenlines']['max_odchylka_pct']} %, "
          f"{M['screenlines']['porovnanych']} porovnaných")
    print(f"verdikt              : {M['verdikt']['overall_pass']} "
          f"(zdroj: {M['verdikt']['zdroj']})")

ZHODA S POZOROVANÍM


,metrika,kalibračná,holdout,DP §4,cieľ
0,r2,0.9028,0.6270,0.83,≥ 0.80
1,slope,1.0026,0.7443,0.94,0.85–1.15
2,pct_rmse,30.2000,47.0000,40.40,≤ 35
3,bias_pct,1.5100,11.8000,8.60,≤ 15


n kalibračných bodov : 14  (adekvátnosť holdoutu: thin)
screenlines          : max odchýlka 88.6 %, 21 porovnaných
verdikt              : False (zdroj: holdout)


In [7]:
# ODME — koľko musela pokriviť maticu (3. úroveň porovnania z PLAN.md)
if kalibracia:
    sd = kalibracia.get("seed_deviation", {}) or {}
    M["odme"] = {
        "iteracii":            kalibracia.get("iterations"),
        "konvergovalo":        kalibracia.get("converged"),
        "metoda":              kalibracia.get("method"),
        "drift_od_seedu_pct":  sd.get("prior_drift_pct"),
        "buniek_nad_prahom":   sd.get("n_cells_exceeding_threshold"),
        "max_pomer_bunky":     sd.get("max_cell_ratio"),
        "seed_spolu":          sd.get("total_seed"),
        "final_spolu":         sd.get("total_final"),
    }
    hist = kalibracia.get("history", []) or []
    if hist:
        M["odme"]["Z_start"] = hist[0].get("Z_objective")
        M["odme"]["Z_koniec"] = min(h.get("Z_objective", float("inf")) for h in hist)
    print("ODME")
    display(pd.Series(M["odme"]).to_frame("hodnota"))
    print("\nČím menší |drift_od_seedu_pct| a menej buniek nad prahom, tým lepší bol vstupný odhad.")

ODME


,hodnota
iteracii,22
konvergovalo,False
metoda,entropy_odme
drift_od_seedu_pct,None
buniek_nad_prahom,None
max_pomer_bunky,None
seed_spolu,None
final_spolu,None
Z_start,26409580.5
Z_koniec,2454995.2



Čím menší |drift_od_seedu_pct| a menej buniek nad prahom, tým lepší bol vstupný odhad.


---
## 5. Baseline sieťové KPI

In [8]:
def vht(df, vol="wd_daily_tot", t="Congested_Time_Max"):
    if vol not in df.columns or t not in df.columns:
        return None
    v = pd.to_numeric(df[vol], errors="coerce").fillna(0)
    tt = pd.to_numeric(df[t], errors="coerce").fillna(0)
    return float((v * tt / 3600.0).sum())


def vkt(df, vol="wd_daily_tot"):
    if vol not in df.columns or "distance" not in df.columns:
        return None
    v = pd.to_numeric(df[vol], errors="coerce").fillna(0)
    d = pd.to_numeric(df["distance"], errors="coerce").fillna(0) / 1000.0
    return float((v * d).sum())


voc = pd.to_numeric(linky.get("VOC_max"), errors="coerce").fillna(0)
M["baseline_kpi"] = {
    "vht_voz_h":        round(vht(linky) or 0, 1),
    "vkt_voz_km":       round(vkt(linky) or 0, 1),
    "pretazenych_hran": int((voc > 1.0).sum()),
    "objem_spolu":      round(float(pd.to_numeric(linky.get("wd_daily_tot"),
                                                  errors="coerce").fillna(0).sum()), 0),
}
if "LOS_max" in linky.columns:
    M["baseline_kpi"]["los"] = linky["LOS_max"].value_counts().to_dict()

display(pd.Series({k: v for k, v in M["baseline_kpi"].items()
                   if not isinstance(v, dict)}).to_frame("hodnota"))
if "los" in M["baseline_kpi"]:
    print("\nRozdelenie LOS:", M["baseline_kpi"]["los"])

# uložiť objemy na linkoch (bez geometrie — tá je veľká)
linky.drop(columns=["_coords"]).to_parquet(OUT / "links_baseline.parquet", index=False)
print(f"\n  → links_baseline.parquet ({len(linky):,} hrán)")

,hodnota
vht_voz_h,9595749.7
vkt_voz_km,6266210.1
pretazenych_hran,259.0
objem_spolu,62716276.0



Rozdelenie LOS: {'A': 32736, 'B': 1887, 'C': 946, 'D': 398, 'F': 259, 'E': 209}

  → links_baseline.parquet (36,435 hrán)


---
## 6. Scenáre uzávierky

Púšťajú sa dva:

- **celá ulica** (`SCENAR_ULICA`) — hlavný výsledok
- **null scenár** (`SCENAR_NULL_LINK`) — hrana s nulovým objemom; jej „uzávierka" nemôže
  nič presunúť, takže výsledná redistribúcia je **odhad šumu** z opakovaného priraďovania.
  Od skutočného efektu sa má odpočítať.

> `link_id` nie sú stabilné medzi prestavbami siete — scenár definuj názvom ulice, nie ID.


In [ ]:
def spusti_scenar(link_ids, lanes_map=None, typ="full", zostane=0):
    """Spustí scenár a počká naň. Vráti job id."""
    lanes_map = lanes_map or {}
    payload = {"links": [{
        "link_id": int(l), "direction": "both", "closure_type": typ,
        "lanes": max(int(lanes_map.get(int(l), 2)), 1),
        "lanes_remaining": int(zostane),
    } for l in link_ids]}
    r = requests.post(f"{API_URL}/api/scenarios/run", json=payload, timeout=60)
    if r.status_code not in (200, 202):
        raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
    jid = r.json()["id"]
    t0 = time.time()
    while time.time() - t0 < 3600:
        st = requests.get(f"{API_URL}/api/scenarios/{jid}/status", timeout=15).json()
        print(f"\r  {st['status']} ({st['elapsed_seconds']:.0f}s)   ", end="")
        if st["status"] == "done":
            print(); return jid
        if st["status"] == "error":
            raise RuntimeError(st.get("error"))
        time.sleep(3)
    raise TimeoutError


def num(df, col):
    return pd.to_numeric(df.get(col), errors="coerce").fillna(0)


def kpi_scenara(res, zavrete):
    dv = num(res, "delta_vol")
    bv, sv = num(res, "baseline_vol"), num(res, "wd_daily_tot")
    ct = num(res, "Congested_Time_Max") / 3600.0
    zav = res["link_id"].isin(zavrete)
    return {
        "zavretych_hran": len(zavrete),
        "objem_na_zavretych_pred": round(float(bv[zav].sum())),
        "objem_na_zavretych_po": round(float(sv[zav].sum())),
        "hran_nad_500": int((dv.abs() > 500).sum()),
        "hran_nad_100": int((dv.abs() > 100).sum()),
        "redistribucia": round(float(dv.abs().sum())),
        "max_narast": round(float(dv.max())),
        "max_pokles": round(float(dv.min())),
        "vht_scenar": round(float((sv * ct).sum()), 1),
        "vkt_scenar": round(float((sv * num(res, "distance") / 1000).sum()), 1),
    }


SCENARE = {}

# --- null scenár: odhad šumu ---
try:
    jid = spusti_scenar([SCENAR_NULL_LINK])
    res = geojson_to_df(get_json(f"/api/scenarios/{jid}/results"))
    SCENARE["null"] = {"res": res, "zavrete": [SCENAR_NULL_LINK],
                       "kpi": kpi_scenara(res, [SCENAR_NULL_LINK])}
    k = SCENARE["null"]["kpi"]
    print(f"null scenár (link {SCENAR_NULL_LINK}): objem pred = {k['objem_na_zavretych_pred']:,.0f}")
    print(f"  → šum: redistribúcia {k['redistribucia']:,.0f}, {k['hran_nad_500']:,} hrán > 500")
except Exception as e:
    print("null scenár zlyhal:", e)


In [ ]:
# --- hlavný scenár: celá ulica ---
ulica = linky[linky["name"].astype(str).str.contains(SCENAR_ULICA, na=False)]
if "link_type" in ulica.columns and (ulica["link_type"] == "secondary").any():
    ulica = ulica[ulica["link_type"] == "secondary"]
ids = ulica["link_id"].astype(int).tolist()
lanes_map = (dict(zip(ulica["link_id"].astype(int), num(ulica, "lanes").replace(0, 2).astype(int)))
             if "lanes" in ulica.columns else {})

print(f"{SCENAR_ULICA}: {len(ids)} hrán, "
      f"{num(ulica, 'distance').sum():,.0f} m, "
      f"objem {num(ulica, 'wd_daily_tot').sum():,.0f} voz/deň")

if ids:
    try:
        jid = spusti_scenar(ids, lanes_map=lanes_map)
        res = geojson_to_df(get_json(f"/api/scenarios/{jid}/results"))
        SCENARE[SCENAR_ULICA] = {"res": res, "zavrete": ids, "kpi": kpi_scenara(res, ids)}
        print(f"scenár {SCENAR_ULICA}: hotovo")
    except Exception as e:
        print(f"scenár {SCENAR_ULICA} zlyhal: {e}")
else:
    print("Ulica sa v sieti nenašla — skontroluj SCENAR_ULICA.")


In [ ]:
M["scenare"] = {k: v["kpi"] for k, v in SCENARE.items()}

for meno, s in SCENARE.items():
    bez = meno.replace(" ", "_").replace("/", "_")
    s["res"].drop(columns=["_coords"], errors="ignore").to_parquet(
        OUT / f"scenario_{bez}.parquet", index=False)
    print(f"  → scenario_{bez}.parquet")

if M["scenare"]:
    t = pd.DataFrame(M["scenare"]).T
    display(t)

    # efekt očistený o šum z null scenára
    if "null" in M["scenare"] and len(M["scenare"]) > 1:
        sum_ = M["scenare"]["null"]
        print("\nEfekt očistený o šum (null scenár):")
        for meno, k in M["scenare"].items():
            if meno == "null":
                continue
            print(f"  {meno}: redistribúcia {k['redistribucia']:,.0f} "
                  f"− šum {sum_['redistribucia']:,.0f} "
                  f"= {k['redistribucia'] - sum_['redistribucia']:,.0f} voz/deň")
            print(f"    hrán > 500: {k['hran_nad_500']:,} − {sum_['hran_nad_500']:,} "
                  f"= {k['hran_nad_500'] - sum_['hran_nad_500']:,}")


---
## 7. Mapy

In [ ]:
def fmt(x):
    """Číslo s medzerou ako oddeľovačom tisícov (5 655, nie 5,655)."""
    try:
        return f"{float(x):,.0f}".replace(",", "\u00a0")
    except Exception:
        return "?"


def stred(df):
    la, lo = [], []
    for c in df["_coords"].dropna().head(400):
        for x, y in (c[::10] or c):
            la.append(y); lo.append(x)
    return (sum(la) / len(la), sum(lo) / len(lo)) if la else (49.195, 16.606)


def sirka(v, vmax, lo=1.0, hi=9.0):
    return lo if not vmax or pd.isna(v) else lo + (hi - lo) * min(abs(v) / vmax, 1.0)


def legenda(m, nadpis, polozky, poznamka=""):
    """polozky = [(farba, popis), ...]"""
    riadky = "".join(
        f'<div style="margin:2px 0"><span style="display:inline-block;width:26px;height:4px;'
        f'background:{f_};vertical-align:middle;margin-right:6px"></span>{p}</div>'
        for f_, p in polozky)
    pozn = f'<div style="margin-top:6px;color:#666;font-size:11px">{poznamka}</div>' if poznamka else ""
    html = (f'<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:#fff;'
            f'padding:10px 12px;border:1px solid #bbb;border-radius:6px;'
            f'font:13px/1.4 system-ui,sans-serif;box-shadow:0 1px 6px rgba(0,0,0,.2)">'
            f'<b>{nadpis}</b>{riadky}{pozn}</div>')
    m.get_root().html.add_child(folium.Element(html))
    return m


LOS_FARBY = {"A": "#2c7fb8", "B": "#41b6c4", "C": "#a1dab4",
             "D": "#fecc5c", "E": "#fd8d3c", "F": "#e31a1c"}

if HAS_FOLIUM:
    vid = linky[num(linky, "wd_daily_tot") >= 1000]
    m = folium.Map(location=stred(vid), zoom_start=12, tiles="CartoDB positron")
    vmax = float(num(vid, "wd_daily_tot").quantile(0.98))
    for _, r in vid.iterrows():
        if not r["_coords"]:
            continue
        folium.PolyLine(
            [(y, x) for x, y in r["_coords"]],
            color=LOS_FARBY.get(str(r.get("LOS_max")), "#999"),
            weight=sirka(r.get("wd_daily_tot", 0), vmax), opacity=0.8,
            tooltip=(f"<b>{r.get('name') or '(bez názvu)'}</b><br>"
                     f"{fmt(r.get('wd_daily_tot', 0))} voz/deň<br>"
                     f"V/C {float(r.get('VOC_max') or 0):.2f} · LOS {r.get('LOS_max')}"),
        ).add_to(m)
    legenda(m, "Denný objem — farba = LOS",
            [(LOS_FARBY[k], f"LOS {k}" + (" (voľno)" if k == "A" else " (kolóna)" if k == "F" else ""))
             for k in "ABCDEF"],
            f"Hrúbka = objem. Zobrazené hrany ≥ 1\u00a0000 voz/deň ({len(vid):,} z {len(linky):,}).".replace(",", "\u00a0"))
    m.save(str(OUT / "mapa_baseline.html"))
    print("  → mapa_baseline.html")
    display(m)


In [ ]:
if HAS_FOLIUM and SCENARE:
    posledna = None
    for meno, s in SCENARE.items():
        res = s["res"]
        d = res[num(res, "delta_vol").abs() >= 100]
        if "low_volume_flag" in d.columns:
            d = d[d["low_volume_flag"] != 1]
        m = folium.Map(location=stred(d if len(d) else res), zoom_start=12,
                       tiles="CartoDB positron")
        vmax = float(num(d, "delta_vol").abs().quantile(0.95)) if len(d) else 1.0
        for _, r in d.iterrows():
            if not r["_coords"]:
                continue
            dv = float(r["delta_vol"])
            folium.PolyLine(
                [(y, x) for x, y in r["_coords"]],
                color="#d7191c" if dv > 0 else "#2c7bb6",
                weight=sirka(dv, vmax, 1.5, 10.0), opacity=0.85,
                tooltip=(f"<b>{r.get('name') or '(bez názvu)'}</b><br>"
                         f"{fmt(r.get('baseline_vol', 0))} → {fmt(r.get('wd_daily_tot', 0))} voz/deň<br>"
                         f"<b>{'+' if dv > 0 else ''}{fmt(dv)}</b> "
                         f"({float(r.get('delta_pct') or 0):+.1f}\u00a0%)"),
            ).add_to(m)
        for _, r in res[res["link_id"].isin(s["zavrete"])].iterrows():
            if r["_coords"]:
                folium.PolyLine([(y, x) for x, y in r["_coords"]],
                                color="#000", weight=11, opacity=0.95,
                                tooltip="uzavreté").add_to(m)
        legenda(m, f"Zmena objemu — {meno}",
                [("#d7191c", "nárast"), ("#2c7bb6", "pokles"), ("#000000", "uzavreté hrany")],
                f"Hrúbka = veľkosť zmeny. Zobrazené |Δ| ≥ 100 voz/deň ({len(d):,} hrán).".replace(",", "\u00a0"))
        bez = meno.replace(" ", "_").replace("/", "_")
        m.save(str(OUT / f"mapa_scenar_{bez}.html"))
        print(f"  → mapa_scenar_{bez}.html")
        posledna = m
    if posledna is not None:
        display(posledna)
else:
    print("Žiadne scenáre na zobrazenie.")


---
## 8. Uloženie

In [14]:
def git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd="..",
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None


cfg_txt = Path("../config/brno/sim.yaml").read_text(encoding="utf-8")

M["_meta"] = {
    "variant": VARIANT,
    "popis": POPIS,
    "vytvorene": datetime.now().isoformat(timespec="seconds"),
    "api_url": API_URL,
    "mesto": meta.get("place_name"),
    "git_commit": git("rev-parse", "--short", "HEAD"),
    "git_dirty": bool(git("status", "--porcelain")),
    "config_sha256": hashlib.sha256(cfg_txt.encode()).hexdigest()[:16],
}

uloz_json(M, "metrics.json")
(OUT / "sim.yaml.snapshot").write_text(cfg_txt, encoding="utf-8")
print("  → sim.yaml.snapshot")

print(f"\n✅ Uložené do {OUT.resolve()}\n")
for p in sorted(OUT.iterdir()):
    print(f"   {p.name:42s} {p.stat().st_size/1024:>9,.0f} kB")

  → metrics.json
  → sim.yaml.snapshot

✅ Uložené do /Users/magdalena.ondruskova/git/personal/Traffic-sim-backend/simulation_for_article/base_model

   calibration_report.json                           30 kB
   links_baseline.parquet                         2,580 kB
   mapa_baseline.html                             5,451 kB
   metrics.json                                       2 kB
   network_summary.json                             564 kB
   obrazky                                            0 kB
   od_summary.json                                    4 kB
   sim.yaml.snapshot                                  3 kB
   validation_report.json                            29 kB


In [15]:
# ═══ POROVNANIE DVOCH VARIANTOV ══════════════════════════════════════
# Spustiť až keď existujú oba priečinky.

A = Path("../simulation_for_article/base_model/metrics.json")
B = Path("../simulation_for_article/updated_version_1/metrics.json")

if A.exists() and B.exists():
    a, b = json.loads(A.read_text()), json.loads(B.read_text())

    riadky = []
    for sekcia, kluce in [
        ("dopyt", ["od_spolu", "commuting", "other", "external_local", "external_through"]),
        ("kalibracia", ["n", "r2", "slope", "pct_rmse", "bias_pct"]),
        ("holdout", ["r2", "slope", "pct_rmse", "bias_pct"]),
        ("odme", ["drift_od_seedu_pct", "buniek_nad_prahom", "Z_koniec"]),
        ("baseline_kpi", ["vht_voz_h", "vkt_voz_km", "pretazenych_hran"]),
    ]:
        for k in kluce:
            va = (a.get(sekcia) or {}).get(k)
            vb = (b.get(sekcia) or {}).get(k)
            if va is None and vb is None:
                continue
            try:
                zmena = round(vb - va, 4)
                zmena_pct = round((vb - va) / abs(va) * 100, 1) if va else None
            except Exception:
                zmena = zmena_pct = None
            riadky.append({"sekcia": sekcia, "metrika": k,
                           "base": va, "updated": vb,
                           "zmena": zmena, "zmena_%": zmena_pct})

    porovnanie = pd.DataFrame(riadky)
    display(porovnanie)
    porovnanie.to_csv("../simulation_for_article/porovnanie.csv", index=False)
    print("→ ../simulation_for_article/porovnanie.csv")
else:
    print("Porovnanie preskočené — chýba jeden z variantov:")
    print(f"  base_model        : {'✓' if A.exists() else '✗'}")
    print(f"  updated_version_1 : {'✓' if B.exists() else '✗'}")

Porovnanie preskočené — chýba jeden z variantov:
  base_model        : ✓
  updated_version_1 : ✗
